# Spotify Artist Data Cleaning & Analytics (5-Task Module)

Welcome! In this module, you will clean and analyze a dirty real-world dataset of global Spotify artists (`spotify_artists.csv`).

### Dataset Issues To Solve:
- Non-standard numeric formats (`"28797,6"`, `"25,574.2"`, `NA`, `null`, `?`)
- Inconsistent text casing (`female` / `F` / `Female`, `US` / `United States` / `USA`)
- Duplicate artist records (`21 Savage` vs `21savage`, `Karol G`)
- Trailing junk columns and missing stream values

---

### Assignment Tasks Overview:
1. **Task 1**: Load `spotify_artists.csv`, normalize column headers, and inspect initial data types.
2. **Task 2**: Convert messy numeric strings into valid float values.
3. **Task 3**: Standardize categorical fields (`Sex`, `country_of_origin`, `primary_genre`, `Artist Type`).
4. **Task 4**: Impute missing total streams and remove duplicate artist records.
5. **Task 5**: Generate summary analytics and export `cleaned_spotify_artists.csv`.

In [1]:
# Task 1: Load Dataset & Inspect Structure
import pandas as pd
import numpy as np

# Load dataset (pre-loaded in workspace)
df = pd.read_csv('spotify_artists.csv')

# Clean column headers (strip spaces, lowercase, replace spaces with underscores)
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_').str.replace('(', '', regex=False).str.replace(')', '', regex=False)

print("Columns in Dataset:", df.columns.tolist())
df.head(10)

Columns in Dataset: ['artist_name', 'sex', 'country_of_origin', 'primary_language', 'primary_genre', 'artist_type', 'debut_year', 'total_streams_m', 'lead_streamsm', 'feature_streams_m']



<div>
<style scoped>
    .dataframe tbody tr th:only-of-type {
        vertical-align: middle;
    }

    .dataframe tbody tr th {
        vertical-align: top;
    }

    .dataframe thead th {
        text-align: right;
    }
</style>
<table border="1" class="dataframe">
  <thead>
    <tr style="text-align: right;">
      <th></th>
      <th>artist_name</th>
      <th>sex</th>
      <th>country_of_origin</th>
      <th>primary_language</th>
      <th>primary_genre</th>
      <th>artist_type</th>
      <th>debut_year</th>
      <th>total_streams_m</th>
      <th>lead_streamsm</th>
      <th>feature_streams_m</th>
    </tr>
  </thead>
  <tbody>
    <tr>
      <th>0</th>
      <td>Olivia Rodrigo</td>
      <td>female</td>
      <td>US</td>
      <td>English</td>
      <td>Pop</td>
      <td>Solo</td>
      <td>2021</td>
      <td>28799.7</td>
      <td>28797,6</td>
      <td>2.2</td>
    </tr>
    <tr>
      <th>1</th>
      <td>Eminem</td>
      <td>Male</td>
      <td>United States</td>

In [6]:
# Task 2: Convert messy numeric strings into valid numeric values

import numpy as np

# Clean debut year
def clean_year(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip().replace("'", "")

    if value == "?" or value == "":
        return np.nan

    if value.isdigit():
        value = int(value)

        if value < 30:
            return 2000 + value
        elif value < 100:
            return 1900 + value

        return value

    return np.nan


df['debut_year'] = df['debut_year'].apply(clean_year)


# Clean stream columns
stream_columns = [
    'total_streams_m',
    'lead_streamsm',
    'feature_streams_m'
]

def clean_stream(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    # Missing/invalid placeholder
    if value in ["?", "", "nan", "None"]:
        return np.nan

    # Decimal comma: 28797,6 → 28797.6
    if ',' in value and '.' not in value:
        value = value.replace(',', '.')

    # Thousands comma: 25,574.2 → 25574.2
    elif ',' in value and '.' in value:
        value = value.replace(',', '')

    return float(value)


for col in stream_columns:
    df[col] = df[col].apply(clean_stream)


print("Numeric cleaning completed.")

print("\nData Types:")
print(df[
    ['debut_year',
     'total_streams_m',
     'lead_streamsm',
     'feature_streams_m']
].dtypes)

print("\nMissing values:")
print(df[
    ['debut_year',
     'total_streams_m',
     'lead_streamsm',
     'feature_streams_m']
].isna().sum())

Numeric cleaning completed.

Data Types:
debut_year           float64
total_streams_m      float64
lead_streamsm        float64
feature_streams_m    float64
dtype: object

Missing values:
debut_year           120
total_streams_m       15
lead_streamsm         23
feature_streams_m     23
dtype: int64



In [3]:
import pandas as pd

df = pd.read_csv('spotify_artists.csv')

print("Dataset loaded successfully!")
print(df.shape)

Dataset loaded successfully!
(120, 10)



In [8]:
print(df.columns.tolist())

['artist_name ', ' Sex', 'country_of_origin', 'Primary Language ', 'primary_genre', 'Artist Type', 'Debut_Year', 'Total Streams (M)', 'Lead Streams(M)', 'feature streams (m)']



In [10]:
# Task 3: Standardize categorical fields

import numpy as np

# 1. Standardize Sex
df[' Sex'] = (
    df[' Sex']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'f': 'Female',
        'female': 'Female',
        'm': 'Male',
        'male': 'Male',
        'nan': np.nan
    })
)

# 2. Standardize Country of Origin
df['country_of_origin'] = (
    df['country_of_origin']
    .astype(str)
    .str.strip()
    .str.lower()
    .replace({
        'us': 'United States',
        'usa': 'United States',
        'u.s.a': 'United States',
        'united states': 'United States',
        'uk': 'United Kingdom',
        'u.k.': 'United Kingdom',
        'united kingdom': 'United Kingdom',
        'colombia': 'Colombia',
        'mexico': 'Mexico',
        'india': 'India',
        'canada': 'Canada',
        'australia': 'Australia',
        'spain': 'Spain',
        'puerto rico': 'Puerto Rico',
        'trinidad and tobago': 'Trinidad and Tobago',
        'barbados': 'Barbados',
        'korea, south': 'South Korea',
        'south korea': 'South Korea',
        'france': 'France',
        'nigeria': 'Nigeria',
        'scotland': 'Scotland',
        'brazil': 'Brazil',
        'nan': np.nan
    })
)

# 3. Standardize Primary Genre
df['primary_genre'] = (
    df['primary_genre']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.title()
    .replace('Nan', np.nan)
)

# 4. Standardize Artist Type
df['Artist Type'] = (
    df['Artist Type']
    .astype(str)
    .str.strip()
    .str.lower()
    .str.title()
    .replace('Nan', np.nan)
)

# Check results
print("Sex:")
print(df[' Sex'].unique())

print("\nCountry of Origin:")
print(df['country_of_origin'].unique())

print("\nPrimary Genre:")
print(df['primary_genre'].unique())

print("\nArtist Type:")
print(df['Artist Type'].unique())

Sex:
['Female' 'Male' 'mixed' nan]

Country of Origin:
['United States' 'Colombia' 'United Kingdom' 'Mexico' 'Puerto Rico'
 'India' 'Canada' 'Australia' 'Spain' 'Trinidad and Tobago' 'Barbados'
 'South Korea' 'France' 'Nigeria' 'Scotland' 'Brazil' nan]

Primary Genre:
['Pop' 'Hip-Hop' 'Reggaeton' 'Regional Mexican' 'Rock' 'Country'
 'Alternative' 'Latin' 'R&B' 'K-Pop' 'Filmi' 'Edm' 'Punjabi Pop'
 'Afrobeats' 'Funk' nan]

Artist Type:
['Solo' 'Group' nan]



In [12]:
# Re-clean numeric columns before Task 4

import numpy as np

def clean_stream(value):
    if pd.isna(value):
        return np.nan

    value = str(value).strip()

    if value in ["?", "", "nan", "None"]:
        return np.nan

    if ',' in value and '.' not in value:
        value = value.replace(',', '.')

    elif ',' in value and '.' in value:
        value = value.replace(',', '')

    return float(value)

stream_columns = [
    'Total Streams (M)',
    'Lead Streams(M)',
    'feature streams (m)'
]

for col in stream_columns:
    df[col] = df[col].apply(clean_stream)

print(df[stream_columns].dtypes)
print("\nMissing values:")
print(df[stream_columns].isna().sum())

Total Streams (M)      float64
Lead Streams(M)        float64
feature streams (m)    float64
dtype: object

Missing values:
Total Streams (M)      15
Lead Streams(M)        23
feature streams (m)    23
dtype: int64



In [13]:
# Task 4: Impute missing Total Streams and remove duplicates

# Missing values before imputation
print("Missing Total Streams before:",
      df['Total Streams (M)'].isna().sum())

# Calculate median
median_streams = df['Total Streams (M)'].median()

# Fill missing values with median
df['Total Streams (M)'] = df['Total Streams (M)'].fillna(median_streams)

# Count duplicates before removal
duplicates_before = df.duplicated().sum()

# Remove duplicate records
df = df.drop_duplicates().copy()

# Verify
print("\nMedian Total Streams:", median_streams)
print("Missing Total Streams after:",
      df['Total Streams (M)'].isna().sum())
print("Duplicates before removal:", duplicates_before)
print("Duplicates after removal:", df.duplicated().sum())
print("\nFinal dataset shape:", df.shape)

Missing Total Streams before: 15

Median Total Streams: 29449.0
Missing Total Streams after: 0
Duplicates before removal: 6
Duplicates after removal: 0

Final dataset shape: (114, 10)



In [14]:
# Task 5: Summary Analytics and Export

# 1. Basic dataset summary
print("Total Artists:", len(df))

print("\nDataset Shape:", df.shape)

print("\nTotal Streams Summary:")
print(df['Total Streams (M)'].describe())

# 2. Genre-wise analytics
print("\nAverage Streams by Genre:")
genre_summary = (
    df.groupby('primary_genre')['Total Streams (M)']
      .agg(['count', 'mean', 'sum'])
      .sort_values('sum', ascending=False)
)

print(genre_summary)

# 3. Top 10 artists by total streams
print("\nTop 10 Artists by Total Streams:")
top_artists = (
    df[['artist_name ', 'Total Streams (M)']]
    .sort_values('Total Streams (M)', ascending=False)
    .head(10)
)

print(top_artists.to_string(index=False))

# 4. Export cleaned dataset
output_file = 'cleaned_spotify_artists.csv'

df.to_csv(output_file, index=False)

print("\nCleaned dataset exported successfully!")
print("File:", output_file)

Total Artists: 114

Dataset Shape: (114, 10)

Total Streams Summary:
count       114.000000
mean      34926.407018
std       18845.963467
min       16566.700000
25%       25181.000000
50%       29449.000000
75%       38581.650000
max      137492.100000
Name: Total Streams (M), dtype: float64

Average Streams by Genre:
                  count          mean        sum
primary_genre                                   
Hip-Hop              27  39027.107407  1053731.9
Pop                  29  35563.724138  1031348.0
Reggaeton            15  40719.346667   610790.2
R&B                   5  48147.800000   240739.0
Rock                  6  33688.283333   202129.7
Latin                 5  30483.640000   152418.2
Regional Mexican      5  26562.460000   132812.3
K-Pop                 4  26918.725000   107674.9
Edm                   4  26633.725000   106534.9
Afrobeats             5  19022.120000    95110.6
Alternative           2  32737.800000    65475.6
Filmi                 2  28793.450000    57